# 01 - Global Country Model

Builds the leakage-safe feature/label panel for all five markets (SPEC.md
section 6-7), runs one expanding walk-forward fold (SPEC.md section 11),
trains every required baseline plus the tree/MLP/transparent-trend models
(SPEC.md section 8), and compares them **on identical out-of-sample dates**
using rank information coefficient and top-quintile hit rate.

Reuses the same local REAL data cache populated by `00_data_audit.ipynb`
(re-running this notebook standalone will download it if missing).

In [1]:
from datetime import date
from pathlib import Path

import pandas as pd

from frtbot.config import load_markets_config, load_research_config
from frtbot.data.cache import DataCache
from frtbot.data.providers import get_provider
from frtbot.data.fx import align_fx_to_index, fetch_fx_series, identity_fx_series, to_thb_price
from frtbot.data.quality import detect_extreme_return_days
from frtbot.features.build import MarketSeries, build_country_feature_panel
from frtbot.labels.country import country_classification_label, country_regression_label
from frtbot.models.dataset import build_country_dataset, select_dates
from frtbot.backtest.splits import generate_walk_forward_folds
from frtbot.models.baselines import EqualWeightModel, RidgeBaselineModel, RuleMomentumModel
from frtbot.models.tree import TreeModel
from frtbot.models.mlp import MLPModel
from frtbot.models.trend import TransparentTrendModel
from frtbot.models.ensemble import combine_ensemble
from frtbot.reporting.ml_metrics import rank_information_coefficient, top_quintile_hit_rate
from frtbot.seed import set_global_seed

pd.set_option("display.width", 140)
set_global_seed(42)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists():
    REPO_ROOT = REPO_ROOT.parent

START = date(2012, 6, 1)
END = date(2026, 7, 31)

markets_config = load_markets_config(REPO_ROOT / "configs" / "markets.example.yml")
research_config = load_research_config(REPO_ROOT / "configs" / "research.yml")
cache = DataCache(root=REPO_ROOT / "data")
provider = get_provider("yfinance")

## Build features, labels, and the long-format modeling panel

In [2]:
market_series: dict[str, MarketSeries] = {}
for m in markets_config.markets:
    df, _ = cache.get_or_fetch(m.key, m.provider_symbol, "ohlcv", provider, START, END)

    # Data-quality gate (SPEC.md section 3): exclude markets with an implausible single-day
    # move - see notebook 00's audit, which flags this same check. Discovered in practice:
    # 1306.T (JP) has a two-day misadjustment (likely an unadjusted stock split) in the
    # upstream Yahoo Finance feed; including it corrupts volatility/return calculations.
    anomalies = detect_extreme_return_days(df["close"])
    if len(anomalies) > 0:
        print(f"EXCLUDING {m.key} ({m.provider_symbol}): price anomaly on {[d.date() for d in anomalies]}")
        continue

    if m.fx_pair is None:
        fx_aligned = identity_fx_series(df.index)
    else:
        fx_entry = markets_config.fx_for(m)
        rate, _ = fetch_fx_series(fx_entry, cache, provider, START, END)
        fx_aligned = align_fx_to_index(rate, df.index)
    market_series[m.key] = MarketSeries(key=m.key, ohlcv=df, fx_rate_aligned=fx_aligned)

print(f"Usable markets after data-quality gate: {sorted(market_series.keys())}")

panel = build_country_feature_panel(market_series)
thb_close = {k: to_thb_price(s.ohlcv["close"], s.fx_rate_aligned) for k, s in market_series.items()}
thb_daily_return = {k: c.pct_change() for k, c in thb_close.items()}

reg_labels = {
    k: country_regression_label(c, research_config.cash_annual_rate, research_config.horizon_trading_days)
    for k, c in thb_close.items()
}
clf_labels = {k: country_classification_label(v) for k, v in reg_labels.items()}

long_df = build_country_dataset(panel, reg_labels, clf_labels)
print("Long-format panel shape:", long_df.shape)
long_df.tail(3)

EXCLUDING JP (1306.T): price anomaly on [datetime.date(2015, 1, 5), datetime.date(2026, 3, 30), datetime.date(2026, 4, 1)]
Usable markets after data-quality gate: ['CN', 'EU', 'TH', 'US']


Long-format panel shape: (14037, 38)


ret_5d   ret_21d   ret_63d  ret_126d  ret_252d  mom_12_1  dist_sma_20  dist_sma_50  dist_sma_100  dist_sma_200  ...  \
date       market                                                                                                                    ...   
2026-07-30 EU      0.017425  0.016787  0.085877  0.085095  0.211349  0.191350     0.009509     0.022856      0.054676      0.085614  ...   
           TH     -0.012218  0.031403  0.101677  0.233816  0.413280  0.370249    -0.004876     0.024606      0.065145      0.155171  ...   
           US      0.004755 -0.006803  0.045000  0.072203  0.180474  0.188560    -0.005220    -0.002996      0.035027      0.064176  ...   

                   stale_rate_63d  thb_ret_21d  thb_ret_63d  thb_ret_252d  local_fx_return_vs_thb  rel_mom_rank  global_risk_regime_z  \
date       market                                                                                                                       
2026-07-30 EU            0.031746     0.028013     0.093056      0.242500                0.004713          0.25              1.210993   
           TH            0.111111     0.031403     0.101677      0.413280                0.000000          1.00              1.210993   
           US            0.000000    -0.000530     0.076198      0.216467               -0.002683          0.50              1.210993   

                   corr_global  label_regression  label_classification  
date       market                                                       
2026-07-30 EU         0.597129               NaN                  <NA>  
           TH         0.586547               NaN                  <NA>  
           US         0.691411               NaN                  <NA>  

[3 rows x 38 columns]

## Walk-forward fold

Uses the frozen research config's expanding walk-forward windows (8y train /
2y val / 1y test, SPEC.md section 11) against the bounded real-data sample.

In [3]:
all_dates = long_df.index.get_level_values("date").unique()
folds = generate_walk_forward_folds(all_dates, research_config.walk_forward)
print(f"{len(folds)} walk-forward fold(s) available from this bounded sample")
for f in folds:
    print(f"  fold {f.index}: train {f.train_dates.min().date()}..{f.train_dates.max().date()} "
          f"({len(f.train_dates)}d) | val {f.val_dates.min().date()}..{f.val_dates.max().date()} "
          f"({len(f.val_dates)}d) | test {f.test_dates.min().date()}..{f.test_dates.max().date()} "
          f"({len(f.test_dates)}d)")

fold = folds[0]
train_df = select_dates(long_df, fold.train_dates)
val_df = select_dates(long_df, fold.val_dates)
test_df = select_dates(long_df, fold.test_dates)
print(f"\nUsing fold 0 for this notebook: train={len(train_df)} val={len(val_df)} test={len(test_df)} rows")

5 walk-forward fold(s) available from this bounded sample
  fold 0: train 2012-06-01..2020-04-30 (2058d) | val 2020-06-01..2022-05-02 (500d) | test 2022-06-01..2023-05-31 (261d)
  fold 1: train 2012-06-01..2021-04-30 (2318d) | val 2021-06-01..2023-05-02 (501d) | test 2023-06-01..2024-05-31 (261d)
  fold 2: train 2012-06-01..2022-05-02 (2579d) | val 2022-06-01..2024-05-02 (501d) | test 2024-06-03..2025-05-30 (259d)
  fold 3: train 2012-06-01..2023-05-02 (2840d) | val 2023-06-01..2025-05-01 (499d) | test 2025-06-02..2026-05-29 (259d)
  fold 4: train 2012-06-01..2024-05-02 (3101d) | val 2024-06-03..2026-04-30 (497d) | test 2026-06-01..2026-07-30 (44d)

Using fold 0 for this notebook: train=7853 val=1901 test=995 rows


## Train every required baseline and model on identical dates

In [4]:
models = {
    "equal_weight": EqualWeightModel(),
    "rule_momentum": RuleMomentumModel(),
    "linear_ridge": RidgeBaselineModel(seed=research_config.seed),
    "tree": TreeModel(seed=research_config.seed),
    "mlp": MLPModel(seed=research_config.seed),
    "transparent_trend": TransparentTrendModel(),
}

test_scores: dict[str, pd.Series] = {}
for name, model in models.items():
    model.fit(train_df, val_df)
    test_scores[name] = model.predict(test_df)

test_forward_return = test_df["label_regression"]

ensemble_result = combine_ensemble(
    {k: test_scores[k] for k in ("tree", "mlp", "transparent_trend")}, research_config.ensemble
)
test_scores["ensemble"] = ensemble_result.score
print("Ensemble weights used:", ensemble_result.weights_used, "excluded:", ensemble_result.excluded)

Ensemble weights used: {'tree': 0.4, 'mlp': 0.3, 'transparent_trend': 0.3} excluded: []


## Model comparison (SPEC.md acceptance criteria #5)

Every model is scored on the **same out-of-sample test dates**. Rank IC and
top-quintile hit rate are the SPEC.md section 11 primary ML metrics scoped to
this five-market cross-section (see `frtbot.reporting.ml_metrics` docstring
for why Brier/regime-stability metrics are deferred to the M3 stock-level
slice, where the larger per-date cross-section makes them more informative).

In [5]:
comparison = pd.DataFrame(
    {
        "rank_ic": {name: rank_information_coefficient(s, test_forward_return) for name, s in test_scores.items()},
        "top_quintile_hit_rate": {name: top_quintile_hit_rate(s, test_forward_return) for name, s in test_scores.items()},
    }
).sort_values("rank_ic", ascending=False)
comparison

D:\code\FRTBOT\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


D:\code\FRTBOT\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


D:\code\FRTBOT\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,rank_ic,top_quintile_hit_rate
tree,0.211105,0.200772
ensemble,0.122881,0.328185
linear_ridge,0.122008,0.328185
transparent_trend,0.059775,0.254826
mlp,0.034749,0.362934
rule_momentum,-0.105405,0.212355
equal_weight,NaN,0.169884
